# Evidencia disponible para el aprendizaje de MemeBot3

## tl;dr

En la captura del 15 de octubre hay **7,792 registros** del
feature store, no operaciones independientes: rechazos y cierres de sombras.
**0 registros** acreditan retorno financiero neto con
el contrato actual. El T0 más reciente es **2026-10-04**.
El principal configurado, sus metadatos y su registro **no están presentes**.
Los 29 metadatos históricos inspeccionados no
acreditan el contrato financiero actual. No se ha entrenado, cargado pesos ni
activado el bot; estos hechos no demuestran la causa histórica de cero compras.

## Context & Methods

Unidad: registro del feature store, no operación ejecutada. Se usa el selector
nativo por partición: Parquet preferido al CSV del mismo nombre, sin descartar
particiones disponibles únicamente en CSV.

### Key Assumptions

Un timestamp parseable no autentica su reloj. Dirección/T0/tipo es una clave
candidata, no una clave primaria garantizada. Una etiqueta de sombra no acredita
retorno financiero neto ni rentabilidad prospectiva. Reejecutar genera una nueva
captura fechada dentro de work, sin reemplazar capturas previas. Se necesitan los
datos privados locales del proyecto y sus dependencias; no se incluyen datos
brutos de operador en una publicación Git.


In [1]:
# Resolve the existing project when opened from docs/audits or the project root.
from pathlib import Path
import sys
project = next(path for path in (Path.cwd(), *Path.cwd().parents)
    if (path/'run_bot.py').is_file() and (path/'config/config.py').is_file())
if str(project) not in sys.path:
    sys.path.insert(0, str(project))


In [2]:
from pathlib import Path
from collections import Counter
import hashlib, json, os, sys
import pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from config.config import CFG, PROJECT_ROOT
from ml.feature_sources import feature_partition_files
from ml.data_contract import normalize_sample_type
from ml.financial_targets import declared_financial_rows, checked_net_return, supported_financial_training
from features.auxiliary_semantics import checked_row_receipt, PROOF_COLUMN
ROOT = PROJECT_ROOT.resolve()
OUT = ROOT/'work/learning-readiness-stage-20261015'
assert PROJECT_ROOT.resolve() == ROOT.resolve()
assert Path(CFG.FEATURES_DIR).resolve().is_relative_to(ROOT.resolve())
assert Path(CFG.MODEL_PATH).resolve().is_relative_to(ROOT.resolve())
sys.dont_write_bytecode = True
def no_operational_io(event, args):
    if event in {'socket.connect', 'socket.connect_ex', 'socket.getaddrinfo', 'subprocess.Popen'}:
        raise RuntimeError('No network/process execution in this read-only audit')
sys.addaudithook(no_operational_io)
def digest(path): return hashlib.sha256(path.read_bytes()).hexdigest()
source_files = sorted(path for path in Path(CFG.FEATURES_DIR).glob('features_*') if path.is_file())
before = {path.name: digest(path) for path in source_files}
selected = feature_partition_files(Path(CFG.FEATURES_DIR))
assert selected, 'No feature source selected by the native loader'
observed_at = pd.Timestamp.now(tz='UTC')
CAPTURE = OUT/'captures'/observed_at.strftime('%Y%m%dT%H%M%S%fZ')
CAPTURE.mkdir(parents=True, exist_ok=False)
plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False})

## Data
### 1. Archivos y población seleccionados

In [3]:
frames, files, schemas = [], [], {}
for path in selected:
    frame = pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
    frames.append(frame)
    schemas[path.name] = {name: str(kind) for name, kind in frame.dtypes.items()}
    files.append({'file': path.relative_to(ROOT).as_posix(), 'rows': len(frame), 'columns': len(frame.columns),
        'mtime_utc': pd.Timestamp.fromtimestamp(path.stat().st_mtime, tz='UTC').isoformat(),
        'sha256': before[path.name]})
frame = pd.concat(frames, ignore_index=True)
t0 = pd.to_datetime(frame.get('timestamp'), utc=True, errors='coerce')
sample_types = frame['sample_type'].map(normalize_sample_type).fillna('unknown')
types = sample_types.value_counts().rename_axis('tipo').reset_index(name='observaciones')
types['porcentaje'] = types['observaciones'] / len(frame) * 100
days = t0.dropna().dt.strftime('%Y-%m-%d').value_counts().sort_index().rename_axis('fecha_UTC').reset_index(name='observaciones')
identities = Counter((str(address), stamp.isoformat(), str(kind))
    for address, stamp, kind in zip(frame['address'], t0, sample_types)
    if pd.notna(address) and pd.notna(stamp))
profile = {'observed_at_utc': observed_at.isoformat(), 'scope': 'Read-only native feature-source and JSON metadata audit; no weights loaded or fit',
    'source_rows': len(frame), 'selected_files': files, 'schemas': schemas,
    'min_original_t0': t0.min().isoformat(), 'max_original_t0': t0.max().isoformat(),
    'unparseable_original_t0': int(t0.isna().sum()), 'future_original_t0': int(t0.gt(observed_at).sum()),
    'candidate_key_duplicate_excess': sum(value - 1 for value in identities.values()),
    'unique_addresses': int(frame['address'].nunique()), 'types': types.to_dict(orient='records'),
    'days': days.to_dict(orient='records')}
display(pd.DataFrame(files).drop(columns='sha256'))
display(types)

,file,rows,columns,mtime_utc
0,data/features/features_202607.parquet,7779,81,2026-07-15T18:18:25.193220+00:00
1,data/features/features_202610.parquet,13,81,2026-10-04T10:43:35.278763+00:00


,tipo,observaciones,porcentaje
0,policy_reject,5435,69.751027
1,shadow_close,2357,30.248973


### 2. Procedencia financiera y generación de entradas
No se interpreta ninguna etiqueta bruta como beneficio neto. Se ejecutan los
validadores actuales de recibos originales sobre los registros seleccionados.

In [4]:
declared = declared_financial_rows(frame)
checked = sum(checked_net_return(row) is not None for row in frame.loc[declared].to_dict(orient='records'))
current_receipts = 0
for row in frame.to_dict(orient='records'):
    current_receipts += checked_row_receipt(row) is not None
missing = frame.isna().sum()
profile.update(declared_financial_rows=int(declared.sum()), checked_net_rows=int(checked),
    current_auxiliary_strategy_receipt_rows=int(current_receipts),
    column_missingness=[{'column': column, 'missing_rows': int(count), 'missing_rate': float(count / len(frame))}
        for column, count in missing.items()])
display(pd.DataFrame([{'registros': len(frame), 'financieros_declarados': int(declared.sum()),
    'neto_verificable': checked, 'recibo_generacion_actual': current_receipts,
    'duplicados_clave_candidata': profile['candidate_key_duplicate_excess']}]))

,registros,financieros_declarados,neto_verificable,recibo_generacion_actual,duplicados_clave_candidata
0,7792,0,0,0,0


### 3. Principal configurado y artefactos históricos
Solo se leen JSON y se comprueba presencia. No se deserializan pesos ni se
confunde un estado histórico `trained` con un principal activo actual.

In [5]:
model = Path(CFG.MODEL_PATH)
history = []
for path in sorted((ROOT/'ml/models').glob('**/model.meta.json')):
    try:
        metadata = json.loads(path.read_text(encoding='utf-8'))
        history.append({'file': path.relative_to(ROOT).as_posix(),
            'supported_financial_training': supported_financial_training(metadata, entry=True),
            'activation_ready_declared': metadata.get('activation_ready'),
            'last_train_attempt_at': metadata.get('last_train_attempt_at')})
    except (ValueError, TypeError, OSError):
        history.append({'file': path.relative_to(ROOT).as_posix(), 'json_readable': False,
            'supported_financial_training': False})
status_path = ROOT/'data/metrics/train_status.json'
status = json.loads(status_path.read_text(encoding='utf-8')) if status_path.is_file() else {}
profile['current_primary'] = {'model_file': model.relative_to(ROOT).as_posix(),
    'model_exists': model.is_file(), 'metadata_exists': model.with_suffix('.meta.json').is_file(),
    'registry_exists': (ROOT/'ml/model_registry.json').is_file(),
    'gate_mode': str(CFG.ML_GATE_MODE), 'retrain_in_main_loop': bool(CFG.ML_RETRAIN_IN_MAIN_LOOP),
    'model_auto_promote': bool(CFG.MODEL_AUTO_PROMOTE), 'lane_auto_promote': bool(CFG.ML_AUTO_PROMOTE_LANES)}
profile['historical_metadata'] = history
profile['historical_metadata_count'] = len(history)
profile['historical_supported_financial_count'] = sum(item['supported_financial_training'] for item in history)
profile['retained_train_status'] = {key: status.get(key) for key in ('status', 'last_train_attempt_at',
    'last_train_status', 'daemon_updated_at_utc', 'training_scope', 'activation_ready')}
display(pd.DataFrame([profile['current_primary']]))
display(pd.DataFrame([profile['retained_train_status']]))

,model_file,model_exists,metadata_exists,registry_exists,gate_mode,retrain_in_main_loop,model_auto_promote,lane_auto_promote
0,ml/model.pkl,False,False,False,shadow,False,False,False


,status,last_train_attempt_at,last_train_status,daemon_updated_at_utc,training_scope,activation_ready
0,trained,2026-10-04T10:42:21.815222+00:00,trained,2026-10-04T10:42:25Z,productive_strict,False


### 4. Réplicas CSV y señales de actividad
No se suman réplicas CSV con su Parquet. La fecha de modificación de un archivo
o de un log no sustituye al T0 de decisión de las muestras de aprendizaje.

In [6]:
csv = []
for path in sorted(Path(CFG.FEATURES_DIR).glob('features_*.csv')):
    replica = pd.read_csv(path, usecols=lambda name: name in {'timestamp', 'sample_type'})
    replica_t0 = pd.to_datetime(replica.get('timestamp'), utc=True, errors='coerce')
    csv.append({'file': path.relative_to(ROOT).as_posix(), 'rows': len(replica),
        'max_original_t0': replica_t0.max().isoformat(), 'selected_by_native_loader': path in selected,
        'matching_parquet_exists': path.with_suffix('.parquet').is_file()})
profile['csv_replicas'] = csv
profile['recent_log_files'] = [{'file': path.relative_to(ROOT).as_posix(), 'bytes': path.stat().st_size,
    'mtime_utc': pd.Timestamp.fromtimestamp(path.stat().st_mtime, tz='UTC').isoformat()}
    for path in sorted((ROOT/'logs').glob('*'), key=lambda path: path.stat().st_mtime, reverse=True)
    if path.is_file()][:5]
display(pd.DataFrame(csv))

,file,rows,max_original_t0,selected_by_native_loader,matching_parquet_exists
0,data/features/features_202607.csv,7735,2026-07-15T00:25:12.561383+00:00,False,True


## Results
La distribución muestra qué unidad hay realmente en los archivos; no es una
tasa de acierto de operaciones. Las fechas son las presentes en el historial,
sin rellenar días ausentes con una actividad cero inventada.

In [7]:
fig, ax = plt.subplots()
ax.barh(types['tipo'], types['observaciones'], color='#3976A8')
ax.invert_yaxis(); ax.set_xlim(0, types['observaciones'].max() * 1.18)
ax.set_xlabel('Registros guardados; no operaciones ejecutadas')
ax.set_title('Tipos de registros del feature store seleccionado')
for index, count in enumerate(types['observaciones']): ax.text(count + 35, index, f'{count:,}', va='center')
fig.tight_layout(); fig.savefig(CAPTURE/'learning-source-types.png', dpi=160)
display(fig); plt.close(fig)
fig, ax = plt.subplots()
ax.bar(days['fecha_UTC'], days['observaciones'], color='#3976A8')
ax.set_ylim(bottom=0); ax.set_xlabel('T0 original del registro (fecha UTC)')
ax.set_ylabel('Registros'); ax.set_title('Fechas observadas en el feature store seleccionado')
fig.tight_layout(); fig.savefig(CAPTURE/'learning-source-days.png', dpi=160)
display(fig); plt.close(fig)

<Figure size 900x400 with 1 Axes>

<Figure size 900x400 with 1 Axes>

### 5. Reconciliación y conservación

In [8]:
assert sum(item['rows'] for item in files) == len(frame) == int(types['observaciones'].sum())
assert int(days['observaciones'].sum()) + int(t0.isna().sum()) == len(frame)
assert {path.name: digest(path) for path in source_files} == before
assert checked <= int(declared.sum()) <= len(frame)
profile['source_hashes_unchanged'] = True
destination = CAPTURE/'profile.json'
assert not destination.exists(), 'Preserve immutable captured evidence'
destination.write_text(json.dumps(profile, indent=2, allow_nan=False), encoding='utf-8')
print('READINESS_CAPTURE=' + CAPTURE.as_posix())
display(pd.DataFrame([{'T0_no_parseable': int(t0.isna().sum()), 'T0_futuro': int(t0.gt(observed_at).sum()),
    'neto_verificable': checked, 'recibos_actuales': current_receipts, 'archivos_fuente_sin_cambios': True}]))

,T0_no_parseable,T0_futuro,neto_verificable,recibos_actuales,archivos_fuente_sin_cambios
0,0,0,0,0,True


## Takeaways

El historial puede servir para diagnóstico, pero no permite certificar un bot
ganador ni aprobar aprendizaje financiero neto actual. No se deben fabricar
costes, FX, recibos, T0 o cierres para completar las muestras. La configuración
actual desactiva reentrenamiento en el bucle principal y autopromoción; mantener
el runtime apagado es intencional, no una causa demostrada del run histórico.
La validación prospectiva requiere nueva evidencia autorizada y conservada.
